In [2]:
import os
import os.path as osp
import sys
from tqdm import tqdm

import cv2
import numpy as np
import trimesh
import matplotlib.pyplot as plt

from typing import List, Tuple

In [3]:
import plotly.graph_objects as go

In [4]:
sys.path.append("..")

In [5]:
from utils.grasp_utils import (
    convert_4x4_to_7dpose,
    convert_7dpose_to_4x4,
    convert_aligned_to_gripper_pose,
    convert_gripper_to_aligned_pose,
)

from utils.pc_utils import (
    backproject_camera, 
    compute_xyz, 
    load_depth_img,
    filter_outliers
)

# Data Paths

In [ ]:
# k = [554.254691191187, 0.0, 320.5, 0.0, 554.254691191187, 240.5, 0.0, 0.0, 1.0]
# NEW CAM K
k = [527.8869068647631, 0.0, 321.7148665756361, 0.0, 524.7942507494529, 230.2819198622499, 0.0, 0.0, 1.0]
intrinsics = np.array(k).reshape(3, 3)
fx = intrinsics[0, 0]
fy = intrinsics[1, 1]
px = intrinsics[0, 2]
py = intrinsics[1, 2]
print(intrinsics)

In [ ]:
TASKS_DIR = "/home/ninad/Datasets/MMDemo/newCamK"
task_names = sorted(os.listdir(TASKS_DIR))
print(task_names)
TO_SAVE = False

In [8]:
# NOTE: Manual intervation / Errors noticed with these tasks:
# NOTE: Backward (Gripper Open Frames) get bugs due to insufficient object mask!
# Task18 [WARNING] -- gripper opening frame -- manually added correct frame
# Task19 [ERROR] -- file i/o error during processing for obj mask
# Task01 [WARNING] -- gripper opening frame -- manually added correct frame
# Task21 [WARNING] -- gripper opening frame -- manually added correct frame (basically we dont have correct seg frames for all frames!)
# Task22 [WARNING] -- gripper opening frame -- manually added correct frame


In [ ]:
task_id = task_names[2]
print("TASK:", task_id)
input_dir = osp.join(TASKS_DIR, task_id)

# input_dir = "/home/ninad/Datasets/MMDemo/whiteboard-eraser_interval_0.05/"
hamer_root_dir = osp.join(input_dir, "out", "hamer")
hamer_npz_dir = osp.join(hamer_root_dir, "model")
depth_img_dir = osp.join(input_dir, "depth")

In [ ]:
samv2_dir = osp.join(input_dir, "out", "samv2")
# NOTE: Assuming only 1 folder within the samv2 directory
masks_dir = osp.join(samv2_dir, os.listdir(samv2_dir)[0], "obj_masks")
print(masks_dir) 

In [11]:
npz_files = [
    f
    for f in os.listdir(hamer_npz_dir)
    if osp.isfile(osp.join(hamer_npz_dir, f)) and f.lower().endswith((".npz"))
]
if not npz_files:
    raise ValueError(f"No npz files found in {hamer_npz_dir}!....")


In [ ]:
npz_files = sorted(npz_files)
print(npz_files)

In [ ]:
# Keeping the first frame id as 1 instead of 0
first_frame_id = "000001"
depth_img_f = osp.join(depth_img_dir, f"{first_frame_id}.png")
depth_im = load_depth_img(depth_img_f)

mask_f = osp.join(masks_dir, f"{first_frame_id}.png")
mask_im = cv2.imread(mask_f, 0)

# scene_pc = compute_xyz(depth_im, fx, fy, px, py)
obj_pc_first_view = backproject_camera(depth_im, intrinsics, target_mask=mask_im)
print(obj_pc_first_view.shape)

In [14]:
# Function to get the hand - object distance based on:
# Distance between object center and hand palm origin (from aligned pose)
def get_HOdist(obj_pc, RT_gripper, gripper_name="fetch_gripper"):
    pose_7d = convert_4x4_to_7dpose(RT_gripper)
    pose_7d_alg = convert_gripper_to_aligned_pose(pose_7d, gripper_name)
    palm_pos = pose_7d_alg[:3]
    obj_center = np.mean(obj_pc, axis=0)
    # ho_dist = np.sqrt(np.sum((obj_center - palm_pos)**2))
    return np.linalg.norm(obj_center - palm_pos)


In [ ]:
num_frames = int(osp.splitext(npz_files[-1])[0]) + 1
print(num_frames)

In [16]:
# Initialize an array for distances (for each frame)
# Setting a default value of 1.2 meters

dists_first_view = [1.2] * num_frames

for idx, npz_f in enumerate(npz_files):
    frame_id = osp.splitext(npz_f)[0]
    frame_id_idx = int(frame_id)
    # NOTE: Code commented out below is not used, but kept for future reference
    # depth_img_f = osp.join(depth_img_dir, f"{frame_id}.png")
    # depth_im = load_depth_img(depth_img_f)
    # mask_f = osp.join(masks_dir, f"{frame_id}.png")
    # mask_im = cv2.imread(mask_f, 0)
    # obj_pc = backproject_camera(depth_im, intrinsics, target_mask=mask_im)
    ### point_cloud = trimesh.points.PointCloud(obj_pc)
    ### point_cloud.export(f"{frame_id}_objpc.ply")   

    npz_fpath = osp.join(hamer_npz_dir, npz_f)
    npz_data = dict(
        np.load(npz_fpath, allow_pickle=True)
    )  # load the npz as dict to be able to update later
    RT_grippers = npz_data["target_transfer_pose"]
    _dists = []
    # print(frame_id)
    for i in range(RT_grippers.shape[0]):
        hodist = get_HOdist(obj_pc_first_view, RT_grippers[i])
        # print(i, hodist)
        _dists.append(hodist)
    dists_first_view[frame_id_idx] = min(_dists)

dists_first_view = np.asarray(dists_first_view)
indices = list(range(len(dists_first_view)))

In [17]:
# plt.plot(indices, dists, marker='o', linestyle='-', color='b', label='Distance')
# # Add labels and title
# plt.xlabel('Index')
# plt.ylabel('Distance')
# plt.title('Distance vs Index')
# plt.legend()

# # Show the plot
# plt.grid()
# plt.show()

In [ ]:
# Create a Plotly figure
fig = go.Figure()

# Add the data as a line plot with markers
fig.add_trace(go.Scatter(
    x=indices, 
    y=dists_first_view, 
    mode='lines+markers',
    name='Distance',
    line=dict(color='blue'),
    marker=dict(size=8)
))

fig.show()

# Find Gripper Closing Frames

- Iterate over the distances in forward order
- Find local minima over a window of size `k` (here `k` = 2) 

In [ ]:
np.mean(dists_first_view)

In [ ]:
delta = np.mean(dists_first_view)
k = 2
frames_close = []
for i in range(num_frames):
    if dists_first_view[i] > delta:
        continue
    if abs(dists_first_view[i] - dists_first_view[i+1]) > 0.02:
        continue 
    if dists_first_view[i] < dists_first_view[i - k] and dists_first_view[i] < dists_first_view[i + k]:
        frames_close.append(i)
        if len(frames_close) >= 2:
            break
print(frames_close)


In [21]:
# last_frame_id = f"{num_frames-1:06}"
# print(last_frame_id)

# Find Gripper Opening Frames

In [ ]:
# Find out the last frame ID
last_frame_id = f"{num_frames-1:06}"
depth_img_f = osp.join(depth_img_dir, f"{last_frame_id}.png")
depth_im = load_depth_img(depth_img_f)

mask_f = osp.join(masks_dir, f"{last_frame_id}.png")
mask_im = cv2.imread(mask_f, 0)

# scene_pc = compute_xyz(depth_im, fx, fy, px, py)
# NOTE: Getting the object point cloud as seen in the last frame
obj_pc_last_view = backproject_camera(depth_im, intrinsics, target_mask=mask_im)
print(obj_pc_last_view.shape)

In [23]:
# Compute distances as before,
# NOTE: but from the object point cloud seen in last frame
dists_last_view = [1.2] * num_frames

for idx, npz_f in enumerate(npz_files):
    frame_id = osp.splitext(npz_f)[0]
    frame_id_idx = int(frame_id)
    npz_fpath = osp.join(hamer_npz_dir, npz_f)
    npz_data = dict(
        np.load(npz_fpath, allow_pickle=True)
    )  # load the npz as dict to be able to update later
    RT_grippers = npz_data["target_transfer_pose"]
    _dists = []
    # print(frame_id)
    for i in range(RT_grippers.shape[0]):
        hodist = get_HOdist(obj_pc_last_view, RT_grippers[i])
        # print(i, hodist)
        _dists.append(hodist)
    dists_last_view[frame_id_idx] = min(_dists)

dists_last_view = np.asarray(dists_last_view)
indices = list(range(len(dists_last_view)))

In [ ]:
# Create a Plotly figure
fig = go.Figure()

# Add the data as a line plot with markers
fig.add_trace(go.Scatter(
    x=indices, 
    y=dists_last_view, 
    mode='lines+markers',
    name='Distance',
    line=dict(color='blue'),
    marker=dict(size=8)
))

fig.show()

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=indices, 
    y=dists_first_view, 
    mode='lines+markers',
    name='From Initial Object Position',
    line=dict(color='orange'),
    marker=dict(size=8)
))

# Add the data as a line plot with markers
fig.add_trace(go.Scatter(
    x=indices, 
    y=dists_last_view, 
    mode='lines+markers',
    name='From Terminal Object Position',
    line=dict(color='blue'),
    marker=dict(size=8),
))

fig.update_layout(yaxis_title="Hand-Object Distance")
fig.update_layout(xaxis_title="Frame Number")

fig.update_layout(legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.2,
        xanchor="center",
        x=0.5
    ),
    font=dict(
        size=18,
    )
)

fig.show()
# fig.write_image("./viz_results/HO_Dist.png")

In [ ]:
np.mean(dists_last_view)

In [ ]:
delta = np.mean(dists_last_view)
k = 3
frames_open = []
for i in range(num_frames)[::-1]:
    if dists_last_view[i] > delta:
        continue
    if abs(dists_last_view[i] - dists_last_view[i+1]) > 0.02:
        continue 
    if dists_last_view[i] < dists_last_view[i - k] and dists_last_view[i] < dists_last_view[i + k]:
        frames_open.append(i)
        if len(frames_open) >= 2:
            break
print(frames_open)

In [28]:
# Data for task ids taken from: https://github.com/IRVLUTD/mm-demo/blob/2ef6888fe3631553c8feb35e634d5a40ae482c3a/vie/create_combined_ply.py#L97

tasks_with_hand_alias = [
    ("task_20_microwave-open_interval_0.05", 1),
    ("task_12_12s-use-spatula", 0),
    ("task_16_12s-wipe-table-with-towel", 0),
    ("task_1_shelf-bottle_interval_0.05", 0),
    ("task_11_17s-use-basting-brush", 0),
    ("task_18_10s-move-chair", 0),
    ("task_14_15s-pouring", 1),
    ("task_10_15s-use-sponge-scrub", 1),
    ("task_3_14s-close-jar-with-lid", 0),
    ("task_22_water_fill_from_water_cooler", 0),
    ("task_21_whiteboard-eraser_interval_0.05", 0),
    ("task_13_13s-sprinkle-salt", 1),
    ("task_8_17s-use_hammer", 0),
    ("task_6_8s-press-keyboard-key", 0),
    ("task_5_15s-open-folder", 0),
    ("task_9-use-stapler", 1),
    ("task_4_17s-fold-towel", 0),
    ("task_19_fetch-shelf-ycb-red-mug_interval_0.05", 1),
    ("task_17_15s-squeeze-sponge-ball", 0),
    ("task_15_19s-use-knife", 0),
    ("task_7_14s-use-cleaning-brush", 0),
    ("task_2_9s-toggle-light-switch", 0),
]

def find_task_in_alias(task_alias_list: List[Tuple], task_id: str) -> Tuple[int]:
    # Find the the task id in the above list and return a Tuple of:
    # (Left/Right Hand Flag, index)
    find_index = None
    hand_flag = None
    for idx, tup in enumerate(task_alias_list):
        if tup[0] == task_id:
            find_index = idx
            hand_flag = tup[1]
            break
    return (hand_flag, find_index)

# Save Data

In [29]:
import json
json_path = "gripper_open_close_info.json"

if TO_SAVE:
    if os.path.exists(json_path):
        # Load the json file as a dict
        with open(json_path, "r") as jf:
            data = json.load(jf)
    else:
        data = {}
    # Add the data for the current task id as new key
    key = task_id
    hand_flag, _ = find_task_in_alias(tasks_with_hand_alias, task_id)
    print(frames_close[0], frames_open[0])
    data[task_id] = {
        "gripper_close_frame": frames_close[0],
        "gripper_open_frame": frames_open[0],
        "hand_flag": hand_flag,
    }

    # Resave the loaded dict to disk
    with open(json_path, "w") as jf:
        json.dump(data, jf, indent=2)

